<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/02_rag_vector_search/faiss_vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Vector Search with FAISS + LangChain

The **retrieval** half of RAG: turn text into embeddings, store them in a **FAISS** index, and find the most relevant documents for a query, with metadata filtering, similarity scores and MMR retrieval. Then index real **Wikipedia** articles and search them.

Free to run: local Hugging Face embeddings, no API key needed.

In [ ]:
!pip install -qU langchain-community langchain-huggingface faiss-cpu wikipedia

## 1. Embeddings
`all-mpnet-base-v2` maps any text to a 768-dimensional vector; similar meanings land close together.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
dim = len(embeddings.embed_query("hello world"))
print("Embedding dimension:", dim)

## 2. Create an empty FAISS vector store
`IndexFlatL2` does exact nearest-neighbour search by Euclidean distance.

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss.IndexFlatL2(dim),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

## 3. Add documents with metadata
Each document carries a `source` tag so searches can be filtered.

In [ ]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)
document_11 = Document(
    page_content="Open AI comes up with a coding IDE called codex",
    metadata={"source": "X"},
)
# document_12 = Document(
#     page_content=docs,
#     metadata={"source": "wiki"},
# )
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
    document_11,
    docs,
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

In [ ]:
# Delete an item from the vector store
vector_store.delete(ids=[uuids[-1]])
print("Documents in store:", vector_store.index.ntotal)

## 4. Search

In [ ]:
results = vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy",
                                         k=2, filter={"source": "tweet"})
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
# L2 distance: lower = more similar
results = vector_store.similarity_search_with_score("Will it be hot tomorrow?", k=1, filter={"source": "news"})
for res, score in results:
    print(f"* [distance={score:.3f}] {res.page_content} [{res.metadata}]")

In [ ]:
# MMR (maximal marginal relevance) balances relevance and diversity
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 1})
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

## 5. Index real Wikipedia articles
Fetch articles with `WikipediaRetriever`, split them into chunks, add them to the same index, and search.

In [ ]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

wiki = WikipediaRetriever(top_k_results=3, doc_content_chars_max=4000)
wiki_docs = wiki.invoke("Artificial intelligence") + wiki.invoke("OpenAI")
print("Wikipedia pages fetched:", [d.metadata["title"] for d in wiki_docs])

chunks = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80).split_documents(wiki_docs)
for c in chunks:
    c.metadata = {"source": "wikipedia", "title": c.metadata.get("title", "")}
vector_store.add_documents(chunks)
print(f"Added {len(chunks)} Wikipedia chunks → store now holds {vector_store.index.ntotal} vectors")

In [ ]:
for query in ["Who founded OpenAI?", "What is machine learning?", "What are the risks of AI?"]:
    print(f"\n🔎 {query}")
    for doc, score in vector_store.similarity_search_with_score(query, k=2, filter={"source": "wikipedia"}):
        print(f"  [{score:.3f}] ({doc.metadata['title']}) {doc.page_content[:160].strip()}...")